# Student Performance — Exploratory Data Analysis (EDA)

**Student:** Sanika Bavale  
**Project Type:** Minor Project — Exploratory Data Analysis  
**Dataset:** Student Performance Factors — Kaggle  
**Dataset Source:** Kaggle — https://www.kaggle.com/datasets/lainguyn123/student-performance-factors

## Project Objective
To understand, clean, analyze and visualize student-performance data using Python. The analysis focuses on study habits, attendance, previous scores, motivation, resources and other factors related to exam performance.

In [ ]:
# Required libraries
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 1. Load the Dataset
The dataset is downloaded from Kaggle using `kagglehub` so the notebook can locate the CSV automatically.

In [ ]:
# Install kagglehub if needed:
# !pip install kagglehub

import kagglehub

dataset_path = kagglehub.dataset_download("lainguyn123/student-performance-factors")
print("Dataset downloaded to:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
print("CSV files found:", csv_files)

if not csv_files:
    raise FileNotFoundError("No CSV file was found in the downloaded Kaggle dataset.")

df = pd.read_csv(csv_files[0])
print("Dataset loaded successfully.")


In [ ]:
# First few records
df.head()


In [ ]:
# Last few records
df.tail()


## 2. Understand the Dataset

In [ ]:
print("Number of rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)


In [ ]:
# Unique values in each column
unique_summary = pd.DataFrame({
    "Column": df.columns,
    "Unique_Values": [df[col].nunique(dropna=False) for col in df.columns]
})
display(unique_summary)


### Observation
The dataset contains a mixture of numerical and categorical variables. `Exam_Score` is the main performance variable.

## 3. Check and Remove Duplicate Records

In [ ]:
rows_before = len(df)
duplicate_count = df.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

df = df.drop_duplicates().copy()
rows_after = len(df)

print("Rows before removing duplicates:", rows_before)
print("Rows after removing duplicates:", rows_after)
print("Rows removed:", rows_before - rows_after)


## 4. Handle Missing Values

In [ ]:
missing_before = df.isnull().sum().sort_values(ascending=False)

missing_table = pd.DataFrame({
    "Missing_Count": missing_before,
    "Missing_Percentage": (missing_before / len(df) * 100).round(2)
})
display(missing_table)


In [ ]:
# Numerical columns -> median; categorical columns -> mode
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Total missing values after handling:", df.isnull().sum().sum())


**Reason:** Median imputation is less affected by extreme numerical observations, while mode imputation preserves the most common categorical value.

## 5. Clean the Dataset

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove leading/trailing spaces from text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

print("Data types after cleaning:")
display(df.dtypes)

print("\nSample of cleaned data:")
display(df.head())


## 6. Analyze Categorical Columns

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
print("Categorical columns:")
print(categorical_cols)


In [ ]:
selected_cats = [
    c for c in ["Gender", "Parental_Involvement", "Motivation_Level",
                "Internet_Access", "School_Type"] if c in df.columns
]

for col in selected_cats:
    plt.figure(figsize=(7, 4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f"{col} Distribution")
    plt.xlabel(col)
    plt.ylabel("Number of Students")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
    print(f"Observation: The distribution above shows how students are distributed across {col} categories.")


## 7. Analyze Numerical Columns

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[numeric_cols].describe().T)


In [ ]:
selected_nums = [
    c for c in ["Hours_Studied", "Attendance", "Previous_Scores",
                "Sleep_Hours", "Tutoring_Sessions", "Physical_Activity",
                "Exam_Score"] if c in df.columns
]

for col in selected_nums:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
    print(f"Observation: The distribution of {col} shows the range and concentration of student values.")


## 8. Detect and Handle Outliers

In [ ]:
outlier_cols = [
    c for c in ["Hours_Studied", "Attendance", "Previous_Scores",
                "Sleep_Hours", "Exam_Score"] if c in df.columns
]

for col in outlier_cols:
    plt.figure(figsize=(7, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


### Outlier Decision
Outliers are inspected visually first. They are not automatically deleted because an extreme value can represent a genuine student observation. Values are retained unless they are clearly invalid according to the dataset's meaning.

## 9. Univariate Analysis

In [ ]:
if "Exam_Score" in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df["Exam_Score"], kde=True)
    plt.title("Exam Score Distribution")
    plt.xlabel("Exam Score")
    plt.ylabel("Number of Students")
    plt.tight_layout()
    plt.show()
    print("Observation: This graph shows the overall distribution of students' exam scores.")


## 10. Bivariate Analysis

In [ ]:
# Hours Studied vs Exam Score
if {"Hours_Studied", "Exam_Score"}.issubset(df.columns):
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="Hours_Studied", y="Exam_Score")
    plt.title("Hours Studied vs Exam Score")
    plt.xlabel("Hours Studied")
    plt.ylabel("Exam Score")
    plt.tight_layout()
    plt.show()
    print("Observation: The scatter plot helps examine whether study time is associated with exam performance.")


In [ ]:
# Exam Score by Motivation Level
if {"Motivation_Level", "Exam_Score"}.issubset(df.columns):
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="Motivation_Level", y="Exam_Score")
    plt.title("Exam Score by Motivation Level")
    plt.xlabel("Motivation Level")
    plt.ylabel("Exam Score")
    plt.tight_layout()
    plt.show()
    print("Observation: The boxplot compares the distribution of exam scores across motivation levels.")


In [ ]:
# Attendance vs Exam Score
if {"Attendance", "Exam_Score"}.issubset(df.columns):
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="Attendance", y="Exam_Score")
    plt.title("Attendance vs Exam Score")
    plt.xlabel("Attendance (%)")
    plt.ylabel("Exam Score")
    plt.tight_layout()
    plt.show()
    print("Observation: The scatter plot is used to inspect the relationship between attendance and exam score.")


## 11. Multivariate Analysis

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
correlation = numeric_df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
if "Exam_Score" in correlation.columns:
    exam_corr = correlation["Exam_Score"].sort_values(ascending=False)
    display(exam_corr.to_frame("Correlation_with_Exam_Score"))
    print("Observation: Correlation values indicate the strength and direction of linear relationships with Exam_Score.")
    print("Correlation does not by itself prove causation.")


In [ ]:
pair_cols = [
    c for c in ["Hours_Studied", "Attendance", "Previous_Scores",
                "Sleep_Hours", "Exam_Score"] if c in df.columns
]

if len(pair_cols) >= 3:
    sns.pairplot(df[pair_cols].dropna())
    plt.show()


## 12. Key Insights

In [ ]:
def pct(x):
    return round(float(x), 2)

print("KEY INSIGHTS")

if "Exam_Score" in df.columns:
    print(f"1. Average exam score: {pct(df['Exam_Score'].mean())}.")
    print(f"2. Minimum exam score: {pct(df['Exam_Score'].min())}; maximum exam score: {pct(df['Exam_Score'].max())}.")

if {"Hours_Studied", "Exam_Score"}.issubset(df.columns):
    r = df["Hours_Studied"].corr(df["Exam_Score"])
    print(f"3. Correlation between hours studied and exam score: {pct(r)}.")

if {"Attendance", "Exam_Score"}.issubset(df.columns):
    r = df["Attendance"].corr(df["Exam_Score"])
    print(f"4. Correlation between attendance and exam score: {pct(r)}.")

if {"Previous_Scores", "Exam_Score"}.issubset(df.columns):
    r = df["Previous_Scores"].corr(df["Exam_Score"])
    print(f"5. Correlation between previous scores and exam score: {pct(r)}.")

if {"Gender", "Exam_Score"}.issubset(df.columns):
    gender_avg = df.groupby("Gender")["Exam_Score"].mean().sort_values(ascending=False)
    print("6. Average exam score by gender:")
    display(gender_avg.to_frame("Average_Exam_Score"))

if {"Motivation_Level", "Exam_Score"}.issubset(df.columns):
    motivation_avg = df.groupby("Motivation_Level")["Exam_Score"].mean().sort_values(ascending=False)
    print("7. Average exam score by motivation level:")
    display(motivation_avg.to_frame("Average_Exam_Score"))

print("8. Categorical and numerical distributions were examined through charts.")
print("9. Duplicate records and missing values were checked and handled.")
print("10. Correlation analysis was used to identify numerical variables associated with exam performance.")


## 13. Final Conclusion
This EDA project examines student-performance data using Python. The work includes dataset inspection, duplicate checking, missing-value handling, data cleaning, categorical and numerical analysis, outlier inspection, univariate, bivariate and multivariate analysis, and correlation analysis.

The findings help identify patterns between academic performance and factors such as study time, attendance, previous scores and other student-related variables. These findings can be useful for a future machine-learning project. Correlation alone does not establish cause-and-effect relationships.

## 14. Final Cleaned DataFrame

In [ ]:
df_clean = df.copy()
print("Final cleaned dataset shape:", df_clean.shape)
display(df_clean.head())


In [ ]:
# Optional: save the cleaned dataset
df_clean.to_csv("student_performance_cleaned.csv", index=False)
print("Cleaned CSV saved as student_performance_cleaned.csv")


## Submission Checklist
- [x] Project introduction and objective
- [x] Kaggle dataset source
- [x] Dataset loading
- [x] Dataset understanding
- [x] Duplicate checking and removal
- [x] Missing-value handling
- [x] Data cleaning
- [x] Categorical analysis
- [x] Numerical analysis
- [x] Outlier detection
- [x] Univariate analysis
- [x] Bivariate analysis
- [x] Multivariate analysis and correlation heatmap
- [x] Key insights
- [x] Final conclusion
- [x] Final cleaned DataFrame (`df_clean`)
